# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Each of this one row of the data repersents one unique content page for a specific client
### Every row contain metrics over a trailing 90-day window


In [1]:
import pandas as pd
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df.head()



,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [6]:
duplicate_count = df['content_id'].duplicated().sum()
print(f"Number of duplicated rows: {duplicate_count}")
print('Number of unique row:')
print(df.shape[0])
print('Number of columns')
print(df.shape[1])

Number of duplicated rows: 0
Number of unique row:
30000
Number of columns
44


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

# The 44 columns will be classify in distinct bucket
## Features 
### impressions_90d, clicks_90d, pageviews_90d, sessions_90d, users_90d, ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct
### Content Porperties:
### word_count, char_count, content_age_days, days_since_last_update
### Keyword Cntext:
### search_volume, competition, cpc, competition_level, content_type, main_intent.

## Lable / Proxy  -- The answers to predict
### trend_direction, is_declining_label

## Context -- For grouping
### cliend_id, content_id 

## Excluded
### trend_pct, trend_direction -- excluded because they can lead to target lekage
### provider_used, model_used -- excluded because the dont contribute to the page traffic performance


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
# The percentage of missing
missing_percentage = df.isnull().mean() * 100
values = missing_percentage[missing_percentage > 0].sort_values(ascending=False)
print(F'The Percentage of missing values')
print(values)

The Percentage of missing values
provider_used        71.460000
word_count_tier      25.663333
char_count           25.663333
word_count           25.663333
char_count_tier      25.663333
model_used           19.110000
trend_pct            11.293333
competition_level     8.700000
search_volume         8.226667
competition           8.226667
cpc                   8.226667
main_intent           7.913333
scroll_rate           0.416667
dtype: float64


In [13]:
df['content_type'].unique()

<ArrowStringArray>
['keyword article', 'feedly article', 'comparison article']
Length: 3, dtype: str

In [12]:
group = df.groupby('content_type')['search_volume'].apply(lambda x:x.isnull().mean() * 100)
print("Missing search volumn group by content type")
print(group)

Missing search volumn group by content type
content_type
comparison article      0.000000
feedly article        100.000000
keyword article         1.367295
Name: search_volume, dtype: float64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  str    
 1   client_id               30000 non-null  str    
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  str    
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  str    
 7   main_intent             27626 non-null  str    
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   str    
 11  model_used              24267 non-null  str    
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null  int64  
 

In [15]:
max_value = df['ctr'].max()
scroll_rate = df['scroll_rate'].max()
traffic_pct = df['ai_traffic_pct'].max()
over_100_scroll = (df['scroll_rate'] > 100).sum()

print('Max CTR:')
print(max_value)
print("Max Scroll Rate:")
print(scroll_rate)
print("Max Traffic Pct:")
print(traffic_pct)
print('Number of rows with scroll more than 100')
print(over_100_scroll)


Max CTR:
100.0
Max Scroll Rate:
300.0
Max Traffic Pct:
300.0
Number of rows with scroll more than 100
119


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.